# Frame-Rate Experiment

**Question:** can the pipeline handle video that isn't 120 fps — and at what cost?

This matters for anything user-uploaded. Phones shoot 30 or 60 fps; the whole system was built on 120.

### The controlled comparison

`raw/videos/legacy_30fps/` holds re-encodes of the same five matches. Same content, same ground truth, only the frame rate differs. So the 30 fps result can be compared directly against the 120 fps run on identical footage.

### The fix: resample pose, don't rescale constants

The classifier takes a **fixed 97-frame input**. Feeding it 24 frames is not an option. So pose is extracted at native fps and then **interpolated onto a 120 fps grid**, after which every downstream formula is unchanged.

| | 120 fps | 30 fps |
|---|---|---|
| activity-gate stride | 12 | **3** |
| detector stride | 8 | **2** |
| activity pad (0.75 s) | 90 | **22** |
| window / NMS / rally gap | — | **unchanged** (on the resampled grid) |

**Order matters:** interpolate *positions* first, *then* differentiate. Computing velocity at 30 fps and rescaling gives a different and wrong answer, because consecutive-frame displacement spans 4× the time.

### What resampling cannot recover

Interpolation restores units, not information. A stroke's acceleration phase is ~120 ms — 12–18 samples at 120 fps, **3–4 at 30 fps**. The central Phase 1 finding was that block-vs-loop lives in the *shape* of the velocity curve, and three samples do not describe a curve.

**Prediction:** peak wrist speed underestimated (the true peak falls between samples and interpolation cuts the corner), classification degraded most on attack-vs-control, contact timing largely intact since 33 ms granularity sits inside the ±67 ms tolerance.

The point of this notebook is to check that prediction against data.

Runtime ~15 min — 4× fewer frames than the 120 fps run.


## 1 · Mount & install

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import torch
assert torch.cuda.is_available(), "No GPU. Runtime > Change runtime type > T4."
print(f"GPU: {torch.cuda.get_device_name(0)}")
!pip uninstall -y -q onnxruntime onnxruntime-gpu 2>&1 | tail -1
!pip install -q --no-deps rtmlib 2>&1 | tail -1
!pip install -q opencv-python numpy tqdm ultralytics pyarrow 2>&1 | tail -1
!pip install -q "onnxruntime-gpu==1.22.0" 2>&1 | tail -1
import os, glob, site
libs=[]
for sp in site.getsitepackages(): libs+=glob.glob(os.path.join(sp,"nvidia","*","lib"))
if libs: open("/content/_ort_libpath.txt","w").write(":".join(libs))
print("\n  NOW: Runtime > Restart session, then run from cell 3.")

GPU: Tesla T4
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.6/69.6 kB 7.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 65.4/65.4 kB 6.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 86.8/86.8 kB 10.0 MB/s eta 0:00:00

  NOW: Runtime > Restart session, then run from cell 3.


## 2 · Config

In [3]:
BASE      = "/content/drive/MyDrive/tt_coach"
SRC_VIDEO = "/content/drive/MyDrive/tt_coach/raw/videos/legacy_30fps/game_1_30fps.mp4"
TEST_ID   = "game_1_30fps"
REF_ID    = "game_1"          # the 120fps run to compare against

GRID_FPS  = 120               # the rate the models were trained on
ACT_PAD_S, MOTION_THR         = 0.75, 0.02
DET_THR, NMS_GAP, RALLY_GAP_S = 0.60, 30, 1.5

import json, math, shutil, time, os, glob, site, subprocess
from pathlib import Path
import numpy as np, pandas as pd
_lp=Path("/content/_ort_libpath.txt")
if _lp.exists(): os.environ["LD_LIBRARY_PATH"]=_lp.read_text()+":"+os.environ.get("LD_LIBRARY_PATH","")
import cv2, torch, torch.nn as nn, torch.nn.functional as F
from tqdm.auto import tqdm

BASE=Path(BASE); META=BASE/"derived/meta"; CKPT=BASE/"models/checkpoints"
ANALYSED=BASE/"derived/analysed"; OUT=BASE/"outputs"; LOCAL=Path("/content/_work")
LOCAL.mkdir(exist_ok=True); (OUT/"metrics").mkdir(parents=True,exist_ok=True)
dev="cuda"

folds=json.loads((META/"folds.json").read_text())
PRE,NF = folds["window"]["pre"], folds["window"]["n_frames"]
CLASSES=["serve","attack","control","defence"]
TECHS=["block","chop","flick","lob","loop","push","serve","smash"]
L_SHO,R_SHO,L_ELB,R_ELB,L_WRI,R_WRI=5,6,7,8,9,10
L_HIP,R_HIP,L_KNE,R_KNE,L_ANK,R_ANK=11,12,13,14,15,16
FLIP=[(1,2),(3,4),(5,6),(7,8),(9,10),(11,12),(13,14),(15,16)]
CAL=json.loads((META/"calibration.json").read_text())
TEMP=CAL["temperature"]
THRESHOLDS=CAL.get("per_class_thresholds",
                   {"serve":0.25,"attack":0.50,"control":1.01,"defence":1.01})

cap=cv2.VideoCapture(SRC_VIDEO)
SRC_FPS=cap.get(cv2.CAP_PROP_FPS); SRC_N=int(cap.get(cv2.CAP_PROP_FRAME_COUNT)); cap.release()
SCALE=GRID_FPS/SRC_FPS
ACT_STRIDE_SRC=max(1,round(12/SCALE))
DET_STRIDE_SRC=max(1,round(8/SCALE))
ACT_PAD_SRC   =round(ACT_PAD_S*SRC_FPS)

print(f"source  {SRC_FPS:.0f} fps, {SRC_N:,} frames ({SRC_N/SRC_FPS/60:.1f} min)")
print(f"grid    {GRID_FPS} fps   scale x{SCALE:.2f}")
print(f"strides gate {ACT_STRIDE_SRC} (was 12), detector {DET_STRIDE_SRC} (was 8), "
      f"pad {ACT_PAD_SRC} (was 90)")
print(f"unchanged on the grid: window {NF}, contact {PRE}, NMS {NMS_GAP}, "
      f"rally {RALLY_GAP_S}s")

source  30 fps, 22,151 frames (12.3 min)
grid    120 fps   scale x4.00
strides gate 3 (was 12), detector 2 (was 8), pad 22 (was 90)
unchanged on the grid: window 97, contact 60, NMS 30, rally 1.5s


## 3 · Models

In [4]:
class Block(nn.Module):
    def __init__(s,c,d,drop=0.1):
        super().__init__()
        s.c1=nn.Conv1d(c,c,5,padding=2*d,dilation=d); s.c2=nn.Conv1d(c,c,5,padding=2*d,dilation=d)
        s.n1,s.n2=nn.BatchNorm1d(c),nn.BatchNorm1d(c); s.do=nn.Dropout(drop)
    def forward(s,x):
        r=x; x=s.do(F.gelu(s.n1(s.c1(x)))); x=s.do(F.gelu(s.n2(s.c2(x)))); return F.gelu(x+r)
class DetNet(nn.Module):
    def __init__(s,c_in,w=128):
        super().__init__()
        s.stem=nn.Sequential(nn.Conv1d(c_in,w,1),nn.BatchNorm1d(w),nn.GELU())
        s.blocks=nn.Sequential(*[Block(w,d) for d in (1,2,4,8,16,32,64)])
        s.hc,s.hs=nn.Conv1d(w,1,1),nn.Conv1d(w,1,1)
    def forward(s,x):
        z=s.blocks(s.stem(x)); return s.hc(z).squeeze(1), s.hs(z).squeeze(1)
class AttnPool(nn.Module):
    def __init__(s,c):
        super().__init__(); s.score=nn.Conv1d(c,1,1)
    def forward(s,x):
        w=torch.softmax(s.score(x),-1); return torch.cat([(x*w).sum(-1),x.max(-1).values],-1)
class ClsNet(nn.Module):
    def __init__(s,c_in,w=128):
        super().__init__()
        s.stem=nn.Sequential(nn.Conv1d(c_in,w,1),nn.BatchNorm1d(w),nn.GELU())
        s.blocks=nn.Sequential(*[Block(w,d,0.2) for d in (1,2,4,8,16,32)])
        s.pool=AttnPool(w)
        s.trunk=nn.Sequential(nn.Linear(w*2,256),nn.GELU(),nn.Dropout(0.3))
        s.shot,s.tech=nn.Linear(256,4),nn.Linear(256,8)
    def forward(s,x):
        z=s.trunk(s.pool(s.blocks(s.stem(x)))); return s.shot(z), s.tech(z)

dck=torch.load(CKPT/"detector_final.pt",map_location=dev,weights_only=False)
cck=torch.load(CKPT/"classifier_final.pt",map_location=dev,weights_only=False)
DET=DetNet(dck["c_in"]).to(dev); DET.load_state_dict(dck["state"]); DET.eval()
CLS=ClsNet(cck["c_in"]).to(dev); CLS.load_state_dict(cck["state"]); CLS.eval()
DMU,DSD=dck["mu"],dck["sd"]; CMU,CSD=cck["mu"].to(dev),cck["sd"].to(dev)
import onnxruntime as ort
assert "CUDAExecutionProvider" in ort.get_available_providers(), "restart after cell 1b"
from ultralytics import YOLO
from rtmlib import RTMPose
YDET=YOLO(str(BASE/"models/detector/best.pt")); YDET.to("cuda")
PLAYER_CLS=[k for k,v in YDET.names.items() if v.lower()=="player"][0]
TABLE_CLS =[k for k,v in YDET.names.items() if v.lower()=="table"][0]
POSE=RTMPose(onnx_model=("https://download.openmmlab.com/mmpose/v1/projects/"
     "rtmposev1/onnx_sdk/rtmpose-l_simcc-body7_pt-body7_420e-384x288-"
     "3f5a1437_20230504.zip"),model_input_size=(288,384),backend="onnxruntime",device="cuda")
print("models ready")

Creating new Ultralytics Settings v0.0.7 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart#ultralytics-settings.


Downloading: "https://download.openmmlab.com/mmpose/v1/projects/rtmposev1/onnx_sdk/rtmpose-l_simcc-body7_pt-body7_420e-384x288-3f5a1437_20230504.zip" to /root/.cache/rtmlib/hub/checkpoints/rtmpose-l_simcc-body7_pt-body7_420e-384x288-3f5a1437_20230504.zip
100%|██████████| 98.9M/98.9M [00:03<00:00, 29.7MB/s]


load /root/.cache/rtmlib/hub/checkpoints/rtmpose-l_simcc-body7_pt-body7_420e-384x288-3f5a1437_20230504.onnx with onnxruntime backend
models ready


## 4 · Extraction at native fps

Identical to the main pipeline except that both strides are scaled to the source rate, so they still correspond to the same *time* interval.

In [5]:
def resolve(frames,mid_x,conf=0.35):
    out=[]
    for r in YDET.predict(frames,verbose=False,conf=conf):
        d={"left":None,"right":None}
        if r.boxes is not None and len(r.boxes):
            xy=r.boxes.xyxy.cpu().numpy(); cl=r.boxes.cls.cpu().numpy().astype(int)
            pl=xy[cl==PLAYER_CLS]
            if len(pl):
                cx=(pl[:,0]+pl[:,2])/2; ls,rs=pl[cx<mid_x],pl[cx>=mid_x]
                if len(ls): d["left"]=ls[np.argmin((ls[:,0]+ls[:,2])/2)]
                if len(rs): d["right"]=rs[np.argmax((rs[:,0]+rs[:,2])/2)]
        out.append(d)
    return out

def find_table(cap,n,k=9):
    bx=[]
    for f in np.linspace(n*0.1,n*0.9,k).astype(int):
        cap.set(cv2.CAP_PROP_POS_FRAMES,int(f)); ok,fr=cap.read()
        if not ok: continue
        r=YDET.predict(fr,verbose=False,conf=0.35)[0]
        if r.boxes is None or not len(r.boxes): continue
        xy=r.boxes.xyxy.cpu().numpy(); cl=r.boxes.cls.cpu().numpy().astype(int)
        tb=xy[cl==TABLE_CLS]
        if len(tb): bx.append(tb[np.argmax((tb[:,2]-tb[:,0])*(tb[:,3]-tb[:,1]))])
    return np.median(np.stack(bx),0).astype(np.float32) if bx else None

def activity_gate(path,nfr,mid_x):
    cap=cv2.VideoCapture(str(path)); flags,idxs,buf,bidx,prev=[],[],[],[],None
    def flush():
        nonlocal buf,bidx,prev
        for j,d in enumerate(resolve(buf,mid_x)):
            both=d["left"] is not None and d["right"] is not None; mov=True
            if both and prev is not None:
                h=max(d["left"][3]-d["left"][1],1)
                mv=max(abs((d["left"][0]+d["left"][2])/2-prev[0]),
                       abs((d["right"][0]+d["right"][2])/2-prev[1]))/h
                mov=mv>MOTION_THR
            if both: prev=((d["left"][0]+d["left"][2])/2,(d["right"][0]+d["right"][2])/2)
            flags.append(both and mov); idxs.append(bidx[j])
        buf,bidx=[],[]
    pb=tqdm(total=nfr,desc="  gate",leave=False); i=0
    while i<nfr:
        if not cap.grab(): break
        if i%ACT_STRIDE_SRC==0:
            ok,fr=cap.retrieve()
            if ok: buf.append(fr); bidx.append(i)
            if len(buf)>=64: flush()
        i+=1
        if i%1000==0: pb.update(1000)
    if buf: flush()
    pb.close(); cap.release()
    spans=[]
    for k,f in enumerate(flags):
        if not f: continue
        s,e=max(0,idxs[k]-ACT_PAD_SRC),min(nfr-1,idxs[k]+ACT_PAD_SRC)
        if spans and s<=spans[-1][1]+1: spans[-1][1]=max(spans[-1][1],e)
        else: spans.append([s,e])
    return spans

def extract_pose(path,spans,mid_x):
    total=sum(e-s+1 for s,e in spans)
    F_=np.zeros(total,np.int32); KP=np.zeros((total,2,17,2),np.float32)
    SC=np.zeros((total,2,17),np.float32); BX=np.zeros((total,2,4),np.float32)
    DT=np.zeros((total,2),bool); SG=np.zeros(total,np.int32)
    cap=cv2.VideoCapture(str(path)); w=0
    pb=tqdm(total=total,desc="  pose",leave=False)
    for si,(s0,e0) in enumerate(spans):
        cap.set(cv2.CAP_PROP_POS_FRAMES,int(s0)); pos=s0
        while pos<=e0:
            n=min(600,e0-pos+1); frames=[]
            for _ in range(n):
                ok,fr=cap.read()
                frames.append(fr if ok else (frames[-1] if frames else np.zeros((720,1280,3),np.uint8)))
            n=len(frames)
            di=list(range(0,n,DET_STRIDE_SRC)); di+=[] if di[-1]==n-1 else [n-1]
            dets=resolve([frames[i] for i in di],mid_x)
            boxes={}
            for pi,side in enumerate(["left","right"]):
                kn=[(i,b) for i,b in zip(di,[d[side] for d in dets]) if b is not None]
                if not kn: continue
                ki=np.array([a for a,_ in kn],float); kb=np.stack([b for _,b in kn]).astype(float)
                boxes[pi]=np.stack([np.interp(np.arange(n),ki,kb[:,c]) for c in range(4)],1).astype(np.float32)
            for k in range(n):
                bb,who=[],[]
                for pi in (0,1):
                    if pi in boxes:
                        b=boxes[pi][k]; bw,bh=b[2]-b[0],b[3]-b[1]
                        b=np.array([max(0,b[0]-bw*.18),max(0,b[1]-bh*.11),b[2]+bw*.18,b[3]+bh*.045],np.float32)
                        bb.append(b); who.append(pi); BX[w+k,pi]=b; DT[w+k,pi]=True
                if bb:
                    kp,sc=POSE(frames[k],bboxes=np.stack(bb))
                    for j,pi in enumerate(who): KP[w+k,pi]=kp[j]; SC[w+k,pi]=sc[j]
                F_[w+k]=pos+k; SG[w+k]=si
            w+=n; pos+=n; pb.update(n); del frames
    pb.close(); cap.release()
    return dict(frame_idx=F_[:w],seg_id=SG[:w],keypoints=KP[:w],
                scores=SC[:w],boxes=BX[:w],detected=DT[:w])
print("extraction ready")

extraction ready


## 5 · Resample onto the 120 fps grid

**The core of the experiment.** Keypoint *positions* are linearly interpolated onto a 120 fps time base, per segment. Velocity is computed afterwards by differencing the resampled positions, which puts it back in the units the models were trained on.

Confidences and detection flags are nearest-neighbour — interpolating a confidence would invent certainty.

In [6]:
def resample_to_grid(raw, src_fps, grid_fps=GRID_FPS):
    """Native-fps pose -> 120fps grid. Positions interpolated, flags nearest."""
    r = grid_fps/src_fps
    F_src, seg = raw["frame_idx"], raw["seg_id"]
    outF, outS, outKP, outSC, outBX, outDT, outSRC = [], [], [], [], [], [], []
    for s in np.unique(seg):
        m = seg == s
        f = F_src[m].astype(np.float64)
        if len(f) < 2: continue
        g = np.arange(f[0]*r, f[-1]*r + 1)              # grid indices
        src_pos = g/r                                    # back in source frames
        kp, sc = raw["keypoints"][m], raw["scores"][m]
        bx, dt = raw["boxes"][m], raw["detected"][m]
        nk = np.zeros((len(g),2,17,2), np.float32)
        nb = np.zeros((len(g),2,4), np.float32)
        for pi in (0,1):
            for j in range(17):
                for c in (0,1):
                    nk[:,pi,j,c] = np.interp(src_pos, f, kp[:,pi,j,c])
            for c in range(4):
                nb[:,pi,c] = np.interp(src_pos, f, bx[:,pi,c])
        near = np.clip(np.searchsorted(f, src_pos), 0, len(f)-1)   # nearest for flags
        outF.append(g.astype(np.int32)); outS.append(np.full(len(g), s, np.int32))
        outKP.append(nk); outSC.append(sc[near]); outBX.append(nb)
        outDT.append(dt[near]); outSRC.append(src_pos.astype(np.float32))
    return dict(frame_idx=np.concatenate(outF), seg_id=np.concatenate(outS),
                keypoints=np.concatenate(outKP), scores=np.concatenate(outSC),
                boxes=np.concatenate(outBX), detected=np.concatenate(outDT),
                src_frame=np.concatenate(outSRC))

# sanity check on a synthetic swing: does resampling preserve peak speed?
t120 = np.arange(0, 97)
true = 1.6*np.exp(-((t120-48)/7.0)**2)                 # a 120fps velocity profile
pos120 = np.cumsum(true)
for src in (60, 30):
    step = 120//src
    f = t120[::step]; p = pos120[::step]
    back = np.interp(t120, f, p)
    v_true = np.diff(pos120).max()
    v_rs   = np.diff(back).max()
    print(f"  {src:>3} fps -> peak speed {v_rs:.3f} vs true {v_true:.3f}   "
          f"({v_rs/v_true-1:+.1%})")
print("""
  That gap is the information loss, and no amount of interpolation removes it:
  the true peak falls between samples and linear interpolation cuts the corner.""")

   60 fps -> peak speed 1.584 vs true 1.600   (-1.0%)
   30 fps -> peak speed 1.493 vs true 1.600   (-6.7%)

  That gap is the information loss, and no amount of interpolation removes it:
  the true peak falls between samples and linear interpolation cuts the corner.


## 6 · Run the pipeline on the resampled stream

In [7]:
def canon(kp,sc,seg,mirror):
    kp=kp.astype(np.float32).copy()
    hip=(kp[:,L_HIP]+kp[:,R_HIP])/2; sho=(kp[:,L_SHO]+kp[:,R_SHO])/2
    torso=np.linalg.norm(sho-hip,axis=-1); scale=np.ones(len(kp),np.float32)
    for s in np.unique(seg):
        m=seg==s; t=torso[m]; t=t[t>1]; scale[m]=np.median(t) if t.size else 1.
    kp=(kp-hip[:,None,:])/np.maximum(scale,1e-3)[:,None,None]
    if mirror:
        kp[...,0]*=-1; sc=sc.copy()
        for a,b in FLIP: kp[:,[a,b]]=kp[:,[b,a]]; sc[:,[a,b]]=sc[:,[b,a]]
    return kp,sc,scale

def build_stream(raw,table):
    seg=raw["seg_id"]; KP=raw["keypoints"]; SC=raw["scores"]; DT=raw["detected"]
    ch,kps,vals,tds=[],[],[],[]
    for pi in (0,1):
        kp,sc,scale=canon(KP[:,pi],SC[:,pi].astype(np.float32),seg,mirror=(pi==1))
        vel=np.zeros_like(kp); vel[1:]=np.diff(kp,axis=0)      # velocity AFTER resampling
        vel[np.diff(seg,prepend=seg[0])!=0]=0
        ch+=[kp.reshape(len(kp),-1),vel.reshape(len(kp),-1),(sc*DT[:,pi:pi+1]).astype(np.float32)]
        kps.append(kp); vals.append((sc>=.35)&DT[:,pi:pi+1])
        hx=(KP[:,pi,L_HIP,0]+KP[:,pi,R_HIP,0])/2
        edge=table[0] if pi==0 else table[2]
        tds.append(np.abs(hx-edge)/np.maximum(scale,1e-3)
                   if table is not None and table[2]>table[0] else np.zeros(len(kp),np.float32))
    cuts=np.where(np.diff(seg)!=0)[0]+1; b=np.concatenate([[0],cuts,[len(seg)]])
    return dict(X=np.nan_to_num(np.concatenate(ch,1).astype(np.float32)),
                kp=np.stack(kps,1),val=np.stack(vals,1),td=np.nan_to_num(np.stack(tds,1)),
                fidx=raw["frame_idx"],src=raw["src_frame"],scores=SC,detected=DT,
                spans=[(int(b[i]),int(b[i+1])) for i in range(len(b)-1)])

def decode(prob,thr=DET_THR,gap=NMS_GAP):
    idx=np.where(prob>=thr)[0]
    if not len(idx): return np.array([],int)
    pk=[i for i in idx if prob[i]==prob[max(0,i-gap//2):i+gap//2+1].max()]
    pk=sorted(pk,key=lambda i:-prob[i]); keep=[]
    for p in pk:
        if all(abs(p-k)>=gap for k in keep): keep.append(p)
    return np.array(sorted(keep),int)

def angle(a,b,c):
    v1,v2=a-b,c-b
    cs=(v1*v2).sum(-1)/np.maximum(np.linalg.norm(v1,axis=-1)*np.linalg.norm(v2,axis=-1),1e-6)
    return np.degrees(np.arccos(np.clip(cs,-1,1)))

t0=time.time()
local=LOCAL/Path(SRC_VIDEO).name
if not local.exists(): print("copying ..."); shutil.copy(SRC_VIDEO,local)
cap=cv2.VideoCapture(str(local)); W=int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
table=find_table(cap,SRC_N); cap.release()
mid_x=(table[0]+table[2])/2 if table is not None else W/2

spans=activity_gate(local,SRC_N,mid_x)
cov=sum(e-s+1 for s,e in spans)/SRC_N
print(f"gate: {len(spans)} regions, {cov:.0%} active")
raw_src=extract_pose(local,spans,mid_x)
print(f"pose at {SRC_FPS:.0f}fps: {len(raw_src['frame_idx']):,} frames")
raw=resample_to_grid(raw_src,SRC_FPS)
print(f"resampled to {GRID_FPS}fps: {len(raw['frame_idx']):,} frames "
      f"(x{len(raw['frame_idx'])/len(raw_src['frame_idx']):.2f})")

st=build_stream(raw,table)
prob=np.zeros(len(st["X"]),np.float32); sidep=np.zeros(len(st["X"]),np.float32)
with torch.no_grad():
    for a,b in st["spans"]:
        x=torch.tensor(((st["X"][a:b]-DMU)/DSD).T[None],dtype=torch.float32,device=dev)
        lc,ls=DET(x)
        prob[a:b]=torch.sigmoid(lc)[0].cpu().numpy(); sidep[a:b]=torch.sigmoid(ls)[0].cpu().numpy()
peaks=decode(prob); sides=["right" if sidep[i]>=0.5 else "left" for i in peaks]
print(f"contacts: {len(peaks)}")

wins,kps,vals,sels=[],[],[],[]
for i,s in zip(peaks,sides):
    sel=np.clip(np.arange(i-PRE,i-PRE+NF),0,len(st["X"])-1)
    pi=0 if s=="left" else 1
    kp=st["kp"][sel,pi]; val=st["val"][sel,pi].astype(np.float32)
    vel=np.zeros_like(kp); vel[1:]=np.diff(kp,axis=0)
    wins.append(np.nan_to_num(np.concatenate([kp.reshape(NF,-1),vel.reshape(NF,-1),
                val,st["td"][sel,pi][:,None]],1)).T.astype(np.float32))
    kps.append(kp); vals.append(val); sels.append(sel)
with torch.no_grad():
    lo,lt=CLS((torch.tensor(np.stack(wins),device=dev)-CMU)/CSD)
    pr=torch.softmax(lo/TEMP,1).cpu().numpy()

src_frames=st["src"][peaks]
ts=src_frames/SRC_FPS                                  # real seconds
rid,sidx,cur,last=[],[],0,None
for t in ts:
    if last is not None and (t-last)>RALLY_GAP_S: cur+=1; k=0
    else: k=0 if last is None else sidx[-1]+1
    rid.append(cur); sidx.append(k); last=t

rows=[]
for n,(i,s) in enumerate(zip(peaks,sides)):
    kp,val=kps[n],vals[n]; wri=R_WRI if s=="left" else L_WRI
    vel=np.zeros_like(kp); vel[1:]=np.diff(kp,axis=0)
    spd=np.linalg.norm(vel,axis=-1); spd[~val.astype(bool)]=np.nan
    w=spd[:,wri]; h=-kp[...,1]
    d_hip=np.linalg.norm(kp[:,wri],axis=-1)
    sh,el=(R_SHO,R_ELB) if wri==R_WRI else (L_SHO,L_ELB)
    ea=angle(kp[:,sh],kp[:,el],kp[:,wri])
    cl=CLASSES[int(pr[n].argmax())]
    rows.append(dict(rally_id=rid[n],shot_index=sidx[n],player=s,
        frame_src=float(src_frames[n]),timestamp_s=float(ts[n]),shot_class=cl,
        class_confidence=float(pr[n].max()),abstain=bool(pr[n].max()<THRESHOLDS[cl]),
        peak_wrist_speed=float(np.nanmax(w)) if np.isfinite(w).any() else np.nan,
        backswing_amplitude=float(np.nanmax(d_hip[:PRE])),
        contact_height=float(h[PRE,wri]-(h[PRE,L_SHO]+h[PRE,R_SHO])/2),
        elbow_angle=float(ea[PRE]),
        follow_through=float(np.nansum(w[PRE:]))))
lo_df=pd.DataFrame(rows)
el=time.time()-t0
print(f"\n{len(lo_df)} shots, {lo_df.rally_id.nunique()} rallies  "
      f"[{el/60:.1f} min = {el/(SRC_N/SRC_FPS):.1f}x realtime]")
lo_df.to_parquet(OUT/"metrics/fps_experiment_shots.parquet",index=False)

copying ...


  gate:   0%|          | 0/22151 [00:00<?, ?it/s]

gate: 59 regions, 53% active


  pose:   0%|          | 0/11798 [00:00<?, ?it/s]

pose at 30fps: 11,798 frames
resampled to 120fps: 47,015 frames (x3.98)
contacts: 159

159 shots, 28 rallies  [17.9 min = 1.5x realtime]


## 7 · Compare against the 120 fps run

Same match, same ground truth. Shots are matched by **timestamp** (±0.15 s), since frame indices are not comparable across frame rates.

In [8]:
def load(stem):
    p=META/f"{stem}.parquet"
    return pd.read_parquet(p) if p.exists() else pd.read_csv(META/f"{stem}.csv")
strokes=load("strokes")
gt=strokes[strokes.video_id==REF_ID]
gt_ts=(gt.frame_120/120).values
hi=pd.read_parquet(ANALYSED/REF_ID/"shots.parquet")

print("="*70); print(f"{'':<22}{'120 fps':>12}{'30 fps':>12}{'delta':>12}")
print("="*70)
print(f"{'shots detected':<22}{len(hi):>12}{len(lo_df):>12}{len(lo_df)-len(hi):>+12}")
print(f"{'vs ground truth':<22}{len(hi)/len(gt)-1:>+11.1%}{len(lo_df)/len(gt)-1:>+12.1%}")
print(f"{'rallies':<22}{hi.rally_id.nunique():>12}{lo_df.rally_id.nunique():>12}")
print(f"{'coachable':<22}{(~hi.abstain).mean():>11.0%}{(~lo_df.abstain).mean():>12.0%}")

# --- detection vs ground truth, by timestamp ---
def match_ts(pred_ts, true_ts, tol=0.067):
    used,tp=set(),0
    for p in pred_ts:
        best,bd=None,tol+1e-9
        for j,t in enumerate(true_ts):
            if j in used: continue
            d=abs(p-t)
            if d<=tol and d<bd: best,bd=j,d
        if best is not None: used.add(best); tp+=1
    return tp,len(pred_ts)-tp,len(true_ts)-tp
for name,df in (("120 fps",hi),("30 fps",lo_df)):
    tp,fp,fn=match_ts(df.timestamp_s.values,gt_ts)
    p=tp/max(tp+fp,1); r=tp/max(tp+fn,1)
    print(f"\n  {name} detection @ +/-67ms: P {p:.3f} R {r:.3f} "
          f"F1 {2*p*r/max(p+r,1e-9):.3f}")

# --- class agreement on shots both runs found ---
pairs=[]
for _,a in hi.iterrows():
    d=(lo_df.timestamp_s-a.timestamp_s).abs()
    if len(d) and d.min()<=0.15:
        b=lo_df.loc[d.idxmin()]
        pairs.append((a.shot_class,b.shot_class,a.peak_wrist_speed,
                      b.peak_wrist_speed,a.backswing_amplitude,b.backswing_amplitude,
                      a.class_confidence,b.class_confidence))
P=pd.DataFrame(pairs,columns=["cls120","cls30","spd120","spd30",
                              "bk120","bk30","cf120","cf30"])
print("\n"+"="*70)
print(f"MATCHED SHOTS: {len(P)} of {len(hi)} ({len(P)/max(len(hi),1):.0%})")
print("="*70)
if len(P):
    agree=(P.cls120==P.cls30).mean()
    print(f"  class agreement      {agree:.1%}")
    print(f"  confidence           {P.cf120.mean():.3f} -> {P.cf30.mean():.3f}"
          f"  ({P.cf30.mean()-P.cf120.mean():+.3f})")
    print(f"\n  peak wrist speed     {P.spd120.mean():.3f} -> {P.spd30.mean():.3f}"
          f"  ({P.spd30.mean()/max(P.spd120.mean(),1e-9)-1:+.1%})   <- the prediction")
    print(f"  backswing amplitude  {P.bk120.mean():.3f} -> {P.bk30.mean():.3f}"
          f"  ({P.bk30.mean()/max(P.bk120.mean(),1e-9)-1:+.1%})")
    print("\n  where the labels disagree:")
    dis=P[P.cls120!=P.cls30]
    if len(dis):
        print(pd.crosstab(dis.cls120,dis.cls30).to_string())
    else:
        print("    none")
    print("\n  agreement by 120fps class:")
    for c in CLASSES:
        s=P[P.cls120==c]
        if len(s): print(f"    {c:<9} {len(s):>4} shots, {(s.cls120==s.cls30).mean():.0%} agree")

                           120 fps      30 fps       delta
shots detected                 164         159          -5
vs ground truth             +1.9%       -1.2%
rallies                         29          28
coachable                     85%         52%

  120 fps detection @ +/-67ms: P 0.970 R 0.988 F1 0.978

  30 fps detection @ +/-67ms: P 0.981 R 0.969 F1 0.975

MATCHED SHOTS: 158 of 164 (96%)
  class agreement      96.2%
  confidence           0.822 -> 0.814  (-0.009)

  peak wrist speed     0.193 -> 0.088  (-54.4%)   <- the prediction
  backswing amplitude  1.235 -> 1.234  (-0.0%)

  where the labels disagree:
cls30    attack  control  defence  serve
cls120                                  
attack        0        1        2      0
defence       2        0        0      1

  agreement by 120fps class:
    serve       24 shots, 100% agree
    attack      60 shots, 95% agree
    control     14 shots, 100% agree
    defence     60 shots, 95% agree


## 8 · Verdict

In [9]:
print("="*70)
tp,fp,fn=match_ts(lo_df.timestamp_s.values,gt_ts); r30=tp/max(tp+fn,1)
tp,fp,fn=match_ts(hi.timestamp_s.values,gt_ts);    r120=tp/max(tp+fn,1)
agree=(P.cls120==P.cls30).mean() if len(P) else 0
spd=P.spd30.mean()/max(P.spd120.mean(),1e-9)-1 if len(P) else 0

print(f"  detection recall   {r120:.3f} -> {r30:.3f}   ({r30-r120:+.3f})")
print(f"  class agreement    {agree:.1%}")
print(f"  peak speed drift   {spd:+.1%}")
print("="*70)
if agree>=0.85 and abs(r30-r120)<0.08:
    print("""  30 fps IS USABLE. Detection holds and labels mostly agree, so the
  resampling approach works. Expect somewhat softer kinematics — check the
  peak-speed drift before using them for coaching thresholds.""")
elif agree>=0.70:
    print("""  30 fps IS MARGINAL. Shots are found but labels shift on a meaningful
  fraction. Usable for counting rallies and shots, NOT for stroke-type
  statistics or coaching feedback.""")
else:
    print("""  30 fps IS NOT USABLE with the current models. Too much of the
  velocity-curve shape is gone. Options: require 60fps+, or retrain on
  30fps-derived features so the model learns the coarser representation.""")
print("""
  Next: set SRC_VIDEO to a 60fps re-encode and re-run to find the floor.
  60fps keeps 6-9 samples in a stroke's acceleration phase versus 3-4 at 30,
  so it should sit much closer to the 120fps baseline.""")

  detection recall   0.988 -> 0.969   (-0.019)
  class agreement    96.2%
  peak speed drift   -54.4%
  30 fps IS USABLE. Detection holds and labels mostly agree, so the
  resampling approach works. Expect somewhat softer kinematics — check the
  peak-speed drift before using them for coaching thresholds.

  Next: set SRC_VIDEO to a 60fps re-encode and re-run to find the floor.
  60fps keeps 6-9 samples in a stroke's acceleration phase versus 3-4 at 30,
  so it should sit much closer to the 120fps baseline.


---
## What this tells you

The comparison is controlled — same match, same annotations, only the frame rate differs — so any difference is attributable to frame rate rather than content.

**Three numbers matter:**

1. **Detection recall** — can it still find the shots?
2. **Class agreement** — does it still label them the same way?
3. **Peak-speed drift** — how much kinematic fidelity is lost, which bounds any coaching use

If 30 fps fails, the same notebook answers the 60 fps question by pointing `SRC_VIDEO` at a 60 fps re-encode.
